# Risk & Return Measures

From-scratch implementations of all major risk and return metrics used in portfolio management.

**Outline**
1. Why one number is not enough
2. Holding period return
3. Arithmetic vs geometric mean return
4. Annualization
5. Sharpe ratio
6. Sortino ratio
7. Treynor ratio
8. Jensen's alpha
9. Information ratio
10. Maximum drawdown
11. Value at Risk (VaR)
12. Conditional VaR / Expected Shortfall
13. Comparison dashboard
14. References### CFA Level 1 Coverage

This notebook covers the complete CFA Level 1 risk and return curriculum:

| Topic | Section |
|:------|:--------|
| Holding period return | §3 |
| Arithmetic vs geometric mean, volatility drag | §4 |
| Money-weighted vs time-weighted return | §4a |
| Nominal vs real returns (Fisher equation) | §4b |
| Risk premium & equity risk premium | §4c |
| Risk aversion & utility functions | §4d |
| Annualisation | §5 |
| Sharpe, Sortino, Treynor ratios | §6–8 |
| Jensen's alpha & information ratio | §9–10 |
| Maximum drawdown | §11 |
| Value at Risk & Expected Shortfall | §12–13 |
### The Scaling Rules

| Quantity | Monthly → Annual | Daily → Annual |
|:---------|:---------------:|:--------------:|
| **Return** | $(1 + r_m)^{12} - 1$ | $(1 + r_d)^{252} - 1$ |
| **Volatility** | $\sigma_m \times \sqrt{12}$ | $\sigma_d \times \sqrt{252}$ |
| **Sharpe ratio** | Controversial — see note below | Controversial |

> **Important:** The return annualisation formula uses compounding (exponentiation). The volatility formula uses the square-root-of-time rule, which assumes returns are independent across periods (i.i.d.). If returns are autocorrelated (mean-reverting or trending), the square-root rule can over- or under-estimate annual volatility.


---
## 1. Why One Number Is Not Enough

A single number cannot capture the full risk-return profile of an investment. Consider two funds:

| | Fund X | Fund Y |
|:--|:--|:--|
| Annual return | 12% | 12% |
| Volatility | 20% | 15% |
| Worst month | -18% | -8% |
| Beta | 1.4 | 0.8 |

Same return, but Fund Y is clearly better on every risk dimension. Which metric should you use to compare them? The answer: **it depends on what you care about.**

- **Total risk?** Use the Sharpe ratio.
- **Downside risk only?** Use the Sortino ratio.
- **Systematic risk?** Use the Treynor ratio.
- **Benchmark-relative performance?** Use the Information ratio.
- **Tail risk (worst-case losses)?** Use VaR or CVaR.
- **Drawdown pain?** Use maximum drawdown.

Each metric has a specific purpose, strengths, and limitations. This notebook implements each from scratch, explains when to use it, and shows when it can be misleading.

> **Key Concept:** No single metric tells the whole story. Professional portfolio evaluation always uses multiple risk-return measures to get a complete picture.

> **CFA Exam Tip:** The CFA curriculum covers Sharpe, Sortino, Treynor, Jensen's alpha, Information ratio, VaR, and CVaR. You need to know the formula, interpretation, and limitations of each.### The Zoo of Risk-Return Metrics

Over the decades, practitioners and academics have developed many metrics. Each one captures a different aspect of the risk-return trade-off:

| Category | Metrics | What they capture |
|:---------|:--------|:-----------------|
| **Return measures** | HPR, arithmetic mean, geometric mean, TWR, MWR | How much did you earn? |
| **Total risk-adjusted** | Sharpe ratio, M-squared | Return per unit of total risk |
| **Downside-focused** | Sortino ratio, max drawdown, VaR, CVaR | How bad can it get? |
| **Systematic risk-adjusted** | Treynor ratio, Jensen's alpha | Return per unit of market risk |
| **Active management** | Information ratio, tracking error | Skill relative to benchmark |

This notebook implements ALL of these from scratch. By the end, you will understand not just the formulas, but *when* each metric is the right one to use.


## 2. Setup

We simulate 5 years of monthly returns for three funds with different characteristics, plus a benchmark (market index). This gives us realistic data to compute all metrics.We simulate three distinct fund profiles so we can see how different metrics tell different stories. Fund A is aggressive (high return, high volatility), Fund B is defensive (moderate return, low volatility), and Fund C adds a twist (crashes mixed with strong rallies).

> **Why simulated data?** Using simulated returns lets us control the properties exactly and isolate each concept. The principles apply identically to real-world return series.


In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize, linalg
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

In [ ]:
# Simulate return series for three funds and a benchmark
T = 60  # 60 months = 5 years
rf_monthly = 0.03 / 12  # 3% annual risk-free

# Benchmark (market)
benchmark = rng.normal(0.008, 0.04, T)

# Fund A: high return, high vol -- the "aggressive growth" fund
fund_a = 1.3 * benchmark + rng.normal(0.001, 0.02, T)

# Fund B: moderate return, low vol, negative skew -- the "steady but occasional blowup" fund
fund_b = 0.7 * benchmark + rng.normal(0.002, 0.01, T) - 0.02 * (rng.random(T) < 0.05)

# Fund C: moderate return, positive alpha -- the "skilled manager" fund
fund_c = 0.9 * benchmark + rng.normal(0.003, 0.015, T)

funds = {'Fund A': fund_a, 'Fund B': fund_b, 'Fund C': fund_c}
colors_f = {'Fund A': PRIMARY, 'Fund B': SECONDARY, 'Fund C': TERTIARY}

---
## 3. Holding Period Return (HPR)

### What It Measures

The total return earned over an entire investment period. If you invested $1 at the start, the HPR tells you how much you have at the end.

### The Formula

$$\text{HPR} = \frac{P_{\text{end}} - P_{\text{begin}} + \text{Income}}{P_{\text{begin}}} = \prod_{t=1}^{T}(1 + r_t) - 1$$

where $r_t$ is the return in period $t$.

### Worked Example

If monthly returns are +5%, -3%, +2%:
$$\text{HPR} = (1.05)(0.97)(1.02) - 1 = 1.0385 - 1 = 3.85\%$$

Note: this is NOT 5% - 3% + 2% = 4%. Compounding matters!

### When to Use It
- Evaluating total wealth accumulation over a specific period.
- Comparing buy-and-hold performance of different investments.

### Limitations
- Does not account for the timing or path of returns (a fund that lost 50% then gained 100% has HPR = 0%, but the experience was terrible).

> **CFA Exam Tip:** HPR is computed by compounding (multiplying), not by adding returns. This is a common exam trap.### Multi-Period HPR

When you have sub-period returns, the total HPR is the chain-linked product:

$$\text{HPR} = (1 + r_1)(1 + r_2)\cdots(1 + r_T) - 1$$

**Worked Example:** Monthly returns of +3%, −1%, +2%:

$$\text{HPR} = (1.03)(0.99)(1.02) - 1 = 1.0398 - 1 = 3.98\%$$

Note this is NOT 3% + (−1%) + 2% = 4%. The product is slightly less because of the compounding of the loss.

> **Key Concept:** Losses compound asymmetrically. A −10% followed by +10% does NOT get you back to even: $(0.90)(1.10) = 0.99$, a net loss of 1%. This is why volatility destroys compound returns (the "volatility drag" we'll see shortly).


In [ ]:
def holding_period_return(returns):
    """Cumulative return over the full period."""
    return np.prod(1 + returns) - 1

# Cumulative wealth paths
fig, ax = plt.subplots()
for name, rets in funds.items():
    wealth = np.cumprod(1 + rets)
    ax.plot(np.arange(T), wealth, color=colors_f[name], linewidth=2, label=f"{name} (HPR={holding_period_return(rets)*100:.1f}%)")

wealth_bm = np.cumprod(1 + benchmark)
ax.plot(np.arange(T), wealth_bm, 'k--', linewidth=1.5, label=f"Benchmark (HPR={holding_period_return(benchmark)*100:.1f}%)")

ax.set_xlabel('Month')
ax.set_ylabel('Growth of $1')
ax.set_title('Cumulative Wealth Paths')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 4. Arithmetic vs Geometric Mean Return

Two different ways to summarize average returns, and they answer different questions.

### Arithmetic Mean

$$\bar{r} = \frac{1}{T} \sum_{t=1}^T r_t$$

**What it answers:** "What is the best estimate of NEXT period's expected return?"

### Geometric Mean

$$g = \left(\prod_{t=1}^T (1+r_t)\right)^{1/T} - 1$$

**What it answers:** "What was the actual compound growth rate per period?"

### The Key Relationship: Volatility Drag

$$g \approx \bar{r} - \frac{\sigma^2}{2}$$

The geometric mean is ALWAYS less than or equal to the arithmetic mean. The gap is the **volatility drag** -- higher volatility destroys compound returns. This is why two funds with the same arithmetic mean but different volatilities will produce different wealth outcomes.

### Worked Example

A fund returns +20%, -10%, +20%, -10%:
- Arithmetic mean: $(20 - 10 + 20 - 10)/4 = 5\%$
- Geometric mean: $(1.2 \times 0.9 \times 1.2 \times 0.9)^{0.25} - 1 = (1.1664)^{0.25} - 1 = 3.92\%$

The 1.08% gap is the volatility drag.

> **Key Concept:** The arithmetic mean overstates the actual compound growth rate. The gap (volatility drag) is approximately $\sigma^2/2$. This means volatility is even more costly than it appears -- it literally eats into your compound returns.

> **CFA Exam Tip:** The geometric mean is always less than or equal to the arithmetic mean. Use arithmetic for FORECASTING next period's return. Use geometric for MEASURING historical compound growth. Know the volatility drag formula.### Complete Guide to Return Types (CFA Level 1)

| Return type | Formula | Use case | Key property |
|:-----------|:--------|:---------|:-------------|
| **Holding period** | $(V_T / V_0) - 1$ | Total return over full period | No averaging |
| **Arithmetic mean** | $\frac{1}{T}\sum r_t$ | Forecasting next period's expected return | Unbiased estimator of $E[r]$ |
| **Geometric mean** | $(\prod(1+r_t))^{1/T} - 1$ | Measuring historical compound growth | Always $\leq$ arithmetic mean |
| **Money-weighted (IRR)** | Solves $\sum \frac{CF_t}{(1+r)^t} = 0$ | Investor's actual experience | Affected by cash flow timing |
| **Time-weighted** | $\prod(1+r_t) - 1$ (chain-linked) | Manager performance evaluation | Unaffected by cash flow timing |
| **Harmonic mean** | $T / \sum(1/r_t)$ | Average cost in dollar-cost averaging | Always $\leq$ arithmetic mean |
| **Nominal** | As reported | Quick comparison | Includes inflation |
| **Real** | $(1+r_{nom})/(1+\pi) - 1$ | Purchasing power comparison | Strips out inflation |

> **CFA Exam Tip:** The exam loves asking "which return measure is appropriate?" The answer depends on the question:
> - "What compound rate did the investment grow at?" → **Geometric mean**
> - "What is the best forecast of next year's return?" → **Arithmetic mean**
> - "How did the manager perform?" → **Time-weighted return**
> - "What return did the investor actually earn?" → **Money-weighted return**


In [ ]:
def arithmetic_mean(returns):
    return np.mean(returns)

def geometric_mean(returns):
    return np.prod(1 + returns) ** (1 / len(returns)) - 1

print(f"{'Fund':<10} {'Arith Mean':>12} {'Geom Mean':>12} {'Vol Drag':>12} {'\u03c3\u00b2/2':>12}")
print('-' * 60)
for name, rets in funds.items():
    am = arithmetic_mean(rets)
    gm = geometric_mean(rets)
    drag = am - gm
    var_half = np.var(rets) / 2
    print(f"{name:<10} {am*100:>11.4f}% {gm*100:>11.4f}% {drag*100:>11.4f}% {var_half*100:>11.4f}%")

---
## 4a. Money-Weighted vs Time-Weighted Return (CFA Level 1)

This is one of the most tested topics in CFA Level 1.

### Time-Weighted Return (TWR)

The TWR measures the **compound growth rate of \$1 initially invested**, removing the effect of external cash flows. It answers: *"How well did the manager perform?"*

$$\text{TWR} = \prod_{t=1}^{n}(1 + r_t) - 1$$

where $r_t$ is the return in each sub-period between cash flows.

### Money-Weighted Return (MWR)

The MWR is the **internal rate of return (IRR)** of all cash flows. It answers: *"What return did the investor actually experience?"*

$$\sum_{t=0}^{n} \frac{CF_t}{(1 + \text{MWR})^t} = 0$$

### The Critical Difference

> **Key Concept:** TWR measures **manager skill** (unaffected by cash flow timing). MWR measures **investor experience** (affected by when money was added/withdrawn). If an investor adds money before a bad period, MWR < TWR.

### Worked Example

An investor puts \$100,000 into a fund on Jan 1:
- Jan–Jun: Fund returns +10%. Portfolio = \$110,000
- Jul 1: Investor adds \$50,000. Portfolio = \$160,000
- Jul–Dec: Fund returns −5%. Portfolio = \$152,000

**TWR:** $(1.10)(0.95) - 1 = 4.5\%$

**MWR:** Solve $-100{,}000 - \frac{50{,}000}{(1+r)^{0.5}} + \frac{152{,}000}{(1+r)^1} = 0$ → MWR ≈ 1.2%

The MWR is much lower because \$50,000 was added right before the bad half.

> **CFA Exam Tip:** GIPS (Global Investment Performance Standards) requires **TWR** for reporting — because it isolates manager performance from client cash flow decisions.

Let's implement both and compare:

In [ ]:
def time_weighted_return(sub_period_returns):
    """TWR: chain-link sub-period returns."""
    return np.prod(1 + np.array(sub_period_returns)) - 1

def money_weighted_return(cash_flows, times):
    """MWR: IRR of cash flows using bisection."""
    def npv(r):
        return sum(cf / (1 + r)**t for cf, t in zip(cash_flows, times))
    # Bisection search for IRR
    lo, hi = -0.5, 2.0
    for _ in range(200):
        mid = (lo + hi) / 2
        if npv(mid) > 0:
            lo = mid
        else:
            hi = mid
    return mid

# Worked example from above
sub_returns = [0.10, -0.05]
twr = time_weighted_return(sub_returns)

# Cash flows: -100k at t=0, -50k at t=0.5, +152k at t=1
cf = [-100000, -50000, 152000]
times = [0, 0.5, 1.0]
mwr = money_weighted_return(cf, times)

print(f"Time-Weighted Return:  {twr*100:.2f}%")
print(f"Money-Weighted Return: {mwr*100:.2f}%")
print(f"\nDifference: {(twr - mwr)*100:.2f} percentage points")
print("\nThe TWR is higher because it ignores the poorly-timed $50k deposit.")
print("The MWR reflects the investor's actual experience — hurt by bad timing.")

### Key Takeaway

The 3.3 percentage point gap between TWR and MWR in this example is entirely due to **cash flow timing**. The manager did fine (4.5% TWR), but the investor's experience was poor (1.2% MWR) because they added money at the worst possible time.

This happens in real life: investors tend to chase performance, adding money after good periods and withdrawing after bad ones. As a result, the average investor's MWR is typically *lower* than the fund's reported TWR.

> **Common Mistake:** Don't confuse TWR and MWR on the exam. If the question asks "what was the fund's performance?" → TWR. If it asks "what return did the investor earn?" → MWR.### A Real-World Example: The Dalbar Study

The Dalbar research firm publishes an annual study showing that the average equity fund investor earns significantly less than the market. For example, over the 20 years ending 2023:
- S&P 500 annualised return: ~10%
- Average equity fund investor's return: ~6%

The gap is almost entirely due to **poor timing** — investors buy after rallies and sell after crashes. This is exactly the MWR vs TWR gap playing out on a massive scale.


---
## 4b. Nominal vs Real Returns

### Why It Matters

A 10% return sounds great — unless inflation was 8%. Your *purchasing power* only increased by ~2%.

### The Fisher Equation

$$(1 + r_{\text{nominal}}) = (1 + r_{\text{real}})(1 + \pi)$$

Solving for real return:

$$r_{\text{real}} = \frac{1 + r_{\text{nominal}}}{1 + \pi} - 1$$

Approximation (for small rates): $r_{\text{real}} \approx r_{\text{nominal}} - \pi$

### Worked Example

| Scenario | Nominal | Inflation | Exact real | Approx real |
|:---------|:-------:|:---------:|:----------:|:-----------:|
| Normal | 8% | 3% | 4.85% | 5.00% |
| High inflation | 12% | 9% | 2.75% | 3.00% |
| Deflation | 3% | −1% | 4.04% | 4.00% |

> **Key Concept:** Always compare investments using real returns when inflation differs across periods or countries. A 15% nominal return with 12% inflation is worse than 5% nominal with 1% inflation.

> **CFA Exam Tip:** The approximation $r_{\text{real}} \approx r_{\text{nominal}} - \pi$ is acceptable on the exam for small rates. Use the exact formula when rates are large.

Let's compute real returns for our funds assuming 3% annual inflation:

In [ ]:
# Nominal vs real returns using Fisher equation
inflation_annual = 0.03
inflation_monthly = (1 + inflation_annual)**(1/12) - 1

print(f"Assumed annual inflation: {inflation_annual*100:.1f}%\n")
print(f"{'Fund':<10} {'Nominal (ann)':>14} {'Real (exact)':>14} {'Real (approx)':>14} {'Approx Error':>14}")
print('-' * 70)
for name, rets in funds.items():
    nominal = (1 + np.mean(rets))**12 - 1
    real_exact = (1 + nominal) / (1 + inflation_annual) - 1
    real_approx = nominal - inflation_annual
    error = real_approx - real_exact
    print(f"{name:<10} {nominal*100:>13.2f}% {real_exact*100:>13.2f}% {real_approx*100:>13.2f}% {error*100:>13.4f}%")

### Interpreting the Results

The exact and approximate real returns are very close — the approximation error is typically just a few basis points for realistic return levels. However, the key insight is how much inflation erodes your returns:

> **Common Mistake:** Investors often anchor on nominal returns. A fund returning 10% in a 7% inflation environment is barely growing purchasing power (real return ~3%). Always think in real terms for long-term planning.### The Rule of 72 in Real Terms

The Rule of 72 says your money doubles in roughly $72 / r$ years. But in *real* terms:
- At 8% nominal, 3% inflation → real rate ~5% → doubles purchasing power in ~14 years
- At 8% nominal, 6% inflation → real rate ~2% → doubles purchasing power in ~36 years

Same nominal return, dramatically different wealth-building timelines.


---
## 4c. Risk Premium & Equity Risk Premium

### Decomposing Expected Returns

Every expected return can be broken down:

$$E(R_i) = R_f + \text{Risk Premium}_i$$

The **risk-free rate** ($R_f$) is the return on a theoretically riskless investment (T-bills). The **risk premium** compensates investors for bearing various risks.

### Types of Risk Premiums

| Premium | Compensates for | Typical range |
|:--------|:---------------|:---:|
| **Equity risk premium (ERP)** | Systematic risk of equities vs bonds | 4–7% |
| **Credit spread** | Default risk on corporate bonds | 1–4% |
| **Liquidity premium** | Difficulty of selling quickly | 0.5–2% |
| **Maturity premium** | Interest rate risk from longer duration | 0.5–2% |

### The Equity Risk Premium

$$\text{ERP} = E(R_{\text{equity}}) - R_f$$

The ERP is the single most important input to the CAPM. Historical U.S. ERP ≈ 5–7% (1926–present). Forward-looking estimates tend to be lower (3–5%).

> **Key Concept:** The ERP is the compensation for bearing the **systematic risk** of the entire stock market. It is the foundation of CAPM: $E(R_i) = R_f + \beta_i \times \text{ERP}$.

> **Common Mistake:** The ERP is NOT the total return on stocks — it is the *excess* return over the risk-free rate. If $R_f = 4\%$ and ERP $= 5\%$, then $E(R_m) = 9\%$.

Let's estimate risk premiums for our simulated funds:

In [ ]:
# Risk premium decomposition
rf_annual = (1 + rf_monthly)**12 - 1
print(f"Risk-free rate (annual): {rf_annual*100:.2f}%\n")

print(f"{'Fund':<10} {'Ann Return':>12} {'Risk Premium':>14} {'Beta':>8} {'Implied ERP':>14}")
print('-' * 62)
for name, rets in funds.items():
    ann_ret = (1 + np.mean(rets))**12 - 1
    rp = ann_ret - rf_annual
    beta = np.cov(rets, benchmark)[0, 1] / np.var(benchmark)
    implied_erp = rp / beta if abs(beta) > 0.01 else float('nan')
    print(f"{name:<10} {ann_ret*100:>11.2f}% {rp*100:>13.2f}% {beta:>8.2f} {implied_erp*100:>13.2f}%")

### Interpreting the Risk Premiums

Each fund's risk premium reflects the compensation for its total risk. The **implied ERP** divides the risk premium by beta to estimate what equity risk premium is consistent with the fund's return:

- Higher-beta funds earn higher risk premiums — but after adjusting for beta, the implied ERP should be similar across funds (if CAPM holds)
- Differences in implied ERP suggest either alpha (skill) or that CAPM doesn't fully explain returns### Historical vs Forward-Looking ERP

| Approach | Method | Typical estimate |
|:---------|:-------|:---------------:|
| **Historical** | Average excess return of stocks over T-bills (1926–present) | 5–7% |
| **Survey** | Ask analysts and academics for their estimates | 3–5% |
| **Implied** | Back out from current stock prices using DDM | 3–6% |

> **CFA Exam Tip:** The exam may ask you to compute the required return on a stock using CAPM: $E(R_i) = R_f + \beta_i \times \text{ERP}$. You need to know what value to use for ERP — usually the question will specify it.


---
## 4d. Risk Aversion & Utility Functions (CFA Level 1)

### The Concept of Risk Aversion

Not all investors are alike. Given two investments with the same expected return, a **risk-averse** investor prefers the one with lower risk. Most investors are risk-averse.

| Investor type | Behaviour | Risk premium required |
|:---|:---|:---|
| **Risk-averse** | Prefers certainty; avoids fair gambles | Positive |
| **Risk-neutral** | Indifferent to risk; cares only about expected return | Zero |
| **Risk-seeking** | Prefers uncertainty; would pay to gamble | Negative (rare) |

### The CFA Level 1 Utility Function

$$U = E(R) - \frac{1}{2} A \sigma^2$$

where:
- $U$ = utility (investor satisfaction)
- $E(R)$ = expected return
- $\sigma^2$ = variance of returns
- $A$ = **risk aversion coefficient** (higher = more risk-averse)

> **Key Concept:** This utility function captures the fundamental trade-off: investors like return (increases utility) but dislike variance (decreases utility). The coefficient $A$ determines how much variance an investor will tolerate for a given return.

### Worked Example

| Portfolio | $E(R)$ | $\sigma$ | $\sigma^2$ |
|:----------|:------:|:--------:|:----------:|
| X | 12% | 20% | 0.04 |
| Y | 8% | 10% | 0.01 |

For $A = 3$: $U_X = 0.12 - 0.5(3)(0.04) = 0.06$, $U_Y = 0.08 - 0.5(3)(0.01) = 0.065$ → **Prefers Y** (lower risk wins)

For $A = 1$: $U_X = 0.12 - 0.5(1)(0.04) = 0.10$, $U_Y = 0.08 - 0.5(1)(0.01) = 0.075$ → **Prefers X** (higher return wins)

> **CFA Exam Tip:** The **certainty equivalent return** is the risk-free rate that gives the same utility as the risky portfolio. It equals $U$ itself. If $U = 0.06$, the investor would be indifferent between the risky portfolio and a guaranteed 6% return.

### Indifference Curves

Setting $U$ constant: $E(R) = U + \frac{1}{2}A\sigma^2$. This is an upward-sloping parabola in $(\sigma, E(R))$ space. More risk-averse investors have **steeper** curves — they demand much more return per unit of additional risk.

> **Common Mistake:** Steeper indifference curves mean MORE risk-averse, not less. Think: "I need a LOT more return to accept a little more risk."

Let's compute utility for our three funds and visualise indifference curves:

In [ ]:
# Risk aversion and utility function
def utility(expected_return, variance, A):
    """CFA L1 utility function: U = E(R) - 0.5 * A * sigma^2"""
    return expected_return - 0.5 * A * variance

# Compare portfolios for different risk aversion levels
print(f"{'A':>3}  {'U(Fund A)':>10}  {'U(Fund B)':>10}  {'U(Fund C)':>10}  {'Preferred':>10}")
print('-' * 50)
for A in [1, 2, 3, 4, 6]:
    utils = {}
    for name, rets in funds.items():
        er = np.mean(rets) * 12  # annualised
        var = np.var(rets) * 12   # annualised
        utils[name] = utility(er, var, A)
    best = max(utils, key=utils.get)
    print(f"{A:>3}  {utils['Fund A']:>10.4f}  {utils['Fund B']:>10.4f}  {utils['Fund C']:>10.4f}  {best:>10}")

# Plot indifference curves
fig, ax = plt.subplots(figsize=(10, 6))
sigma_range = np.linspace(0.01, 0.35, 200)
for A in [2, 4, 6]:
    for U_level in [0.02, 0.05, 0.08]:
        er_curve = U_level + 0.5 * A * sigma_range**2
        ax.plot(sigma_range * 100, er_curve * 100, alpha=0.5, 
                color=['steelblue', 'coral', 'seagreen'][[2,4,6].index(A)])
    ax.plot([], [], color=['steelblue', 'coral', 'seagreen'][[2,4,6].index(A)], label=f'A = {A}')

# Plot fund positions
for name, rets in funds.items():
    ann_ret = np.mean(rets) * 12
    ann_vol = np.std(rets) * np.sqrt(12)
    ax.scatter(ann_vol * 100, ann_ret * 100, s=100, zorder=5)
    ax.annotate(name, (ann_vol * 100 + 0.5, ann_ret * 100))

ax.set_xlabel('Annualised Volatility (%)')
ax.set_ylabel('Expected Return (%)')
ax.set_title('Indifference Curves for Different Risk Aversion Levels')
ax.legend()
ax.set_xlim(0, 35)
ax.set_ylim(0, 20)
plt.tight_layout()
plt.show()

### Interpreting the Results

The utility table shows how different investors rank the same three funds:
- **Low $A$ (risk-tolerant):** Prefers the higher-return, higher-volatility fund
- **High $A$ (risk-averse):** Prefers the lower-volatility fund, even at lower return

The indifference curve plot shows:
- **Steeper curves (high $A$):** The investor demands a large return increase for any additional risk
- **Flatter curves (low $A$):** The investor readily accepts more risk for modest return improvement
- The **optimal portfolio** is the tangency point between the highest indifference curve and the efficient frontier (or CAL)

---
## 5. Annualization

Monthly and daily metrics need to be converted to annual terms for comparison.

### Annualizing Returns

$$\text{Annualized return} = (1 + g_{\text{monthly}})^{12} - 1$$

We compound the geometric mean, not the arithmetic mean, because we want the actual annual growth rate.

### Annualizing Volatility

$$\text{Annualized volatility} = \sigma_{\text{monthly}} \times \sqrt{12}$$

The $\sqrt{12}$ factor comes from the assumption that returns are roughly independent across months. (If returns are positively autocorrelated, this understates true annual volatility.)

> **CFA Exam Tip:** Annualizing returns uses compounding ($^{12}$). Annualizing volatility uses $\sqrt{12}$. Do not mix these up -- it is a common exam mistake.

In [ ]:
def annualise_return(monthly_return, periods=12):
    return (1 + monthly_return)**periods - 1

def annualise_vol(monthly_vol, periods=12):
    return monthly_vol * np.sqrt(periods)

print(f"{'Fund':<10} {'Ann Return':>12} {'Ann Vol':>10}")
print('-' * 34)
for name, rets in funds.items():
    ann_r = annualise_return(geometric_mean(rets))
    ann_v = annualise_vol(np.std(rets))
    print(f"{name:<10} {ann_r*100:>11.2f}% {ann_v*100:>9.2f}%")

---
## 6. Sharpe Ratio

### What It Measures

**Excess return per unit of TOTAL risk.** The Sharpe ratio answers: "How much extra return (above the risk-free rate) am I getting for each percentage point of volatility I take on?"

### The Formula

$$\text{SR} = \frac{\bar{r}_P - r_f}{\sigma_P}$$

where:
- $\bar{r}_P$ = average portfolio return
- $r_f$ = risk-free rate
- $\sigma_P$ = standard deviation of portfolio returns

### Interpretation

- **SR > 1.0:** Excellent risk-adjusted return.
- **SR 0.5 - 1.0:** Good.
- **SR 0 - 0.5:** Mediocre -- you are getting some excess return but not much per unit of risk.
- **SR < 0:** Negative -- you earned less than the risk-free rate!

### When to Use It

The Sharpe ratio is the **most widely used** risk-adjusted performance metric. It is appropriate when:
- Comparing portfolios that represent an investor's TOTAL holdings (not sub-portfolios).
- The investor cares about total risk (not just downside or systematic risk).

### When It Is Misleading

1. **Asymmetric return distributions:** The Sharpe ratio assumes returns are normally distributed. If a strategy has high positive skew (lottery-like) or negative skew (frequent small gains, occasional large losses), the Sharpe ratio can be very misleading.
2. **Illiquid assets:** Stale pricing smooths returns, artificially inflating the Sharpe ratio. Private equity and hedge funds often look better than they are.
3. **Leverage:** You can increase the Sharpe ratio by leveraging a strategy, but this does not create genuine value -- it just moves you along the CAL.

### Annualization

$$\text{SR}_{\text{ann}} = \text{SR}_{\text{monthly}} \times \sqrt{12}$$

> **Key Concept:** The Sharpe ratio is excess return per unit of total risk. It is the slope of the Capital Allocation Line from $R_f$ through the portfolio. Higher is better.

> **CFA Exam Tip:** Know that the Sharpe ratio uses TOTAL risk ($\sigma$) in the denominator. For comparing diversified portfolios, it is the right measure. For individual securities within a portfolio, Treynor or alpha may be more appropriate.### When Is the Sharpe Ratio Misleading?

The Sharpe ratio has important limitations:

1. **Assumes normal returns:** If returns are skewed or have fat tails, standard deviation doesn't capture the full risk picture
2. **Penalises upside volatility:** A fund with huge upside swings (good!) gets penalised just as much as one with huge downside swings (bad!)
3. **Can be gamed:** Strategies that sell insurance (collect small premiums, occasionally suffer large losses) can have high Sharpe ratios for years before blowing up
4. **Time-period sensitive:** A fund can have a high Sharpe ratio over 3 years and a low one over 5 years

> **CFA Exam Tip:** The Sharpe ratio is the slope of the Capital Allocation Line (CAL). The portfolio with the highest Sharpe ratio is the **tangency portfolio** — the optimal risky portfolio for any investor.

Let's compute Sharpe ratios for our three funds:


In [ ]:
def sharpe_ratio(returns, rf=0.0):
    excess = returns - rf
    return np.mean(excess) / np.std(excess)

print(f"{'Fund':<10} {'Monthly SR':>12} {'Annualised SR':>14}")
print('-' * 38)
for name, rets in funds.items():
    sr = sharpe_ratio(rets, rf_monthly)
    print(f"{name:<10} {sr:>12.4f} {sr * np.sqrt(12):>14.4f}")

---
## 7. Sortino Ratio

### What It Measures

**Excess return per unit of DOWNSIDE risk.** The Sortino ratio is like the Sharpe ratio, but it only penalizes *negative* deviations from the target return. Upside volatility is not penalized.

### Why This Matters

The Sharpe ratio treats upside and downside volatility equally. But investors do not dislike upside volatility -- they love it! A fund that occasionally has very large positive returns (high upside vol) gets unfairly penalized by the Sharpe ratio. The Sortino ratio fixes this.

### The Formula

$$\text{Sortino} = \frac{\bar{r}_P - r_f}{\sigma_{\text{down}}}$$

where the downside deviation is:

$$\sigma_{\text{down}} = \sqrt{\frac{1}{T}\sum_{t=1}^T \min(r_t - \text{MAR}, 0)^2}$$

- **MAR** = Minimum Acceptable Return (often set equal to $r_f$)
- Only returns BELOW the MAR contribute to the downside deviation.

### Sortino vs Sharpe: When Do They Diverge?

They diverge most when the return distribution is **asymmetric** (skewed):
- **Negative skew** (frequent small gains, rare large losses): Sortino < Sharpe (Sharpe overstates risk-adjusted performance because it underweights the tail risk)
- **Positive skew** (frequent small losses, rare large gains): Sortino > Sharpe (Sharpe penalizes the desirable upside volatility)

> **Key Concept:** The Sortino ratio only penalizes downside risk. It is more appropriate than the Sharpe ratio when returns are not symmetric.

> **CFA Exam Tip:** Know that the Sortino ratio replaces standard deviation with downside deviation. It is more appropriate for strategies with asymmetric returns (e.g., options-based strategies, hedge funds).### Why Downside Risk Matters More

Investors don't lose sleep over *upside* surprises — they worry about *losses*. The Sortino ratio addresses this by only penalising returns below a threshold (the **minimum acceptable return**, or MAR).

**Downside deviation** is computed as:

$$\sigma_D = \sqrt{\frac{1}{T}\sum_{t=1}^T \min(r_t - \text{MAR}, 0)^2}$$

Note: ALL observations are included in the denominator (not just negative ones). This avoids overstating the penalty.

> **Key Concept:** If two funds have the same Sharpe ratio but Fund A has negative skew (occasional large losses) while Fund B has positive skew (occasional large gains), the Sortino ratio will correctly rank Fund B higher.

Let's compare with the Sharpe ratio:


In [ ]:
def sortino_ratio(returns, rf=0.0, mar=None):
    if mar is None:
        mar = rf
    excess = returns - rf
    downside = np.minimum(returns - mar, 0)
    downside_dev = np.sqrt(np.mean(downside**2))
    if downside_dev < 1e-14:
        return np.inf
    return np.mean(excess) / downside_dev

print(f"{'Fund':<10} {'Sharpe':>10} {'Sortino':>10}")
print('-' * 32)
for name, rets in funds.items():
    sr = sharpe_ratio(rets, rf_monthly) * np.sqrt(12)
    so = sortino_ratio(rets, rf_monthly) * np.sqrt(12)
    print(f"{name:<10} {sr:>10.3f} {so:>10.3f}")

---
## 8. Treynor Ratio

### What It Measures

**Excess return per unit of SYSTEMATIC risk (beta).** While the Sharpe ratio uses total risk ($\sigma$), the Treynor ratio uses only beta -- the risk that cannot be diversified away.

### The Formula

$$\text{Treynor} = \frac{\bar{r}_P - r_f}{\beta_P}$$

### When to Use It

Use Treynor when evaluating a portfolio that is **part of a larger diversified portfolio**. In that context, only the systematic risk matters (the unsystematic risk will be diversified away by the other holdings). The Sharpe ratio would overstate the risk because it includes diversifiable risk.

### Limitations

- Meaningless for portfolios with $\beta \leq 0$ (the denominator goes to zero or negative).
- Does not capture unsystematic risk, so inappropriate for undiversified portfolios.

> **CFA Exam Tip:** Treynor uses beta in the denominator (systematic risk). Use it to evaluate sub-portfolios within a larger diversified investment. Use Sharpe for evaluating total portfolios.### Sharpe vs Treynor: When to Use Which

| Metric | Risk measure | Best for |
|:-------|:------------|:---------|
| **Sharpe** | Total risk ($\sigma$) | Evaluating a standalone portfolio |
| **Treynor** | Systematic risk ($\beta$) | Evaluating one fund among many (diversified context) |

> **Key Concept:** If a fund is your *entire* portfolio, use Sharpe (total risk matters). If a fund is *one component* of a diversified portfolio, use Treynor (only systematic risk matters — idiosyncratic risk gets diversified away).

Let's compute Treynor ratios:

> **Common Mistake:** Don't use the Treynor ratio to evaluate an entire portfolio in isolation — it ignores diversifiable risk that the investor still bears. Treynor is designed for evaluating *components* of a larger, diversified portfolio.


In [ ]:
def treynor_ratio(returns, benchmark_returns, rf=0.0):
    excess_p = returns - rf
    excess_m = benchmark_returns - rf
    beta = np.cov(excess_p, excess_m)[0, 1] / np.var(excess_m)
    return np.mean(excess_p) / beta, beta

print(f"{'Fund':<10} {'Beta':>8} {'Treynor':>10}")
print('-' * 30)
for name, rets in funds.items():
    tr, beta = treynor_ratio(rets, benchmark, rf_monthly)
    print(f"{name:<10} {beta:>8.3f} {tr*100:>9.4f}%")

---
## 9. Jensen's Alpha

### What It Measures

The return earned BEYOND what the CAPM predicts for the portfolio's level of systematic risk. Positive alpha = the manager added value. Negative alpha = the manager destroyed value.

### The Formula

$$\alpha_J = \bar{r}_P - [r_f + \beta_P(\bar{r}_M - r_f)]$$

### Interpretation

- $\alpha > 0$: Manager outperformed the CAPM benchmark. Potential evidence of skill.
- $\alpha = 0$: Manager earned exactly what was expected for the risk taken. No skill demonstrated.
- $\alpha < 0$: Manager underperformed. An index fund would have been better.

> **Key Concept:** Alpha is THE measure of active management skill. It strips out the effect of market exposure (beta) to isolate the manager's pure contribution.

> **CFA Exam Tip:** Alpha is the y-intercept of the characteristic line regression. A fund with $\alpha = 2\%$ earned 2% more than CAPM predicted for its beta.### Interpreting Alpha

- $\alpha > 0$: Manager generated value beyond what CAPM predicts — potential skill
- $\alpha = 0$: Return is exactly what CAPM predicts for the level of systematic risk taken
- $\alpha < 0$: Manager underperformed — either bad luck or negative skill

But be careful: a positive alpha might be due to:
- Genuine skill
- Exposure to risk factors not captured by CAPM (size, value, momentum)
- Survivorship bias (we only see funds that didn't close)
- Random chance (even coin-flipping monkeys generate some positive alphas)

> **CFA Exam Tip:** Jensen's alpha is the y-intercept of the regression $R_p - R_f = \alpha + \beta(R_m - R_f) + \varepsilon$. A statistically significant positive alpha suggests skill. The t-statistic of alpha tells you if it's distinguishable from zero.

Let's estimate alpha for each fund:

> **CFA Exam Tip:** Alpha is typically estimated using regression. The t-statistic of the intercept tells you if alpha is statistically significant. With noisy monthly returns, you often need 5+ years of data to detect even a substantial alpha with confidence.


In [ ]:
def jensens_alpha(returns, benchmark_returns, rf=0.0):
    excess_p = returns - rf
    excess_m = benchmark_returns - rf
    beta = np.cov(excess_p, excess_m)[0, 1] / np.var(excess_m)
    alpha = np.mean(excess_p) - beta * np.mean(excess_m)
    return alpha, beta

print(f"{'Fund':<10} {'Alpha (monthly)':>16} {'Alpha (annual)':>15}")
print('-' * 43)
for name, rets in funds.items():
    alpha, _ = jensens_alpha(rets, benchmark, rf_monthly)
    print(f"{name:<10} {alpha*100:>15.4f}% {alpha*12*100:>14.4f}%")

---
## 10. Information Ratio

### What It Measures

**Active return per unit of active risk.** How consistently does the manager outperform the benchmark?

### The Formula

$$\text{IR} = \frac{\bar{r}_P - \bar{r}_B}{\sigma(r_P - r_B)} = \frac{\text{Active return}}{\text{Tracking error}}$$

### Interpretation

- **Active return** = average excess return over the benchmark.
- **Tracking error** = standard deviation of the difference between portfolio and benchmark returns.
- **IR > 0.5:** Good active manager. **IR > 1.0:** Exceptional.

### Why It Matters

A manager might have positive alpha but with wild swings -- sometimes beating the benchmark by 5%, sometimes lagging by 4%. The IR captures this: it rewards consistent outperformance and penalizes erratic performance.

> **Key Concept:** The IR is to active management what the Sharpe ratio is to total return. The Sharpe measures total risk-adjusted return; the IR measures benchmark-relative risk-adjusted return.

> **CFA Exam Tip:** IR = active return / tracking error. It measures the consistency of outperformance. A higher IR means the manager's alpha is more reliable.### What Makes a Good Information Ratio?

| IR | Interpretation |
|:---|:--------------|
| < 0 | Underperforming the benchmark |
| 0.0 – 0.4 | Below average active management |
| 0.4 – 0.7 | Good active management |
| 0.7 – 1.0 | Very good (top quartile) |
| > 1.0 | Exceptional (rare and usually not sustained) |

> **Key Concept:** The IR is the active management equivalent of the Sharpe ratio. Sharpe = excess return over risk-free per unit of total risk. IR = excess return over benchmark per unit of tracking error.

> **CFA Exam Tip:** The **Fundamental Law of Active Management** connects the IR to two factors: $\text{IR} \approx \text{IC} \times \sqrt{\text{BR}}$, where IC = information coefficient (skill) and BR = breadth (number of independent bets). More skill OR more bets → higher IR.

Let's compute IR for our funds:


In [ ]:
def information_ratio(returns, benchmark_returns):
    active = returns - benchmark_returns
    return np.mean(active) / np.std(active)

print(f"{'Fund':<10} {'Active Ret':>12} {'Track Err':>10} {'IR (ann)':>10}")
print('-' * 44)
for name, rets in funds.items():
    active = rets - benchmark
    ir = information_ratio(rets, benchmark)
    print(f"{name:<10} {np.mean(active)*12*100:>11.3f}% {np.std(active)*np.sqrt(12)*100:>9.3f}% {ir*np.sqrt(12):>10.3f}")

---
## 11. Maximum Drawdown

### What It Measures

The **largest peak-to-trough decline** in portfolio value. It answers: "What was the worst possible experience for an investor who bought at the worst time and sold at the worst time?"

### The Formula

$$\text{MDD} = \max_{t} \left( \frac{\text{Peak}_t - \text{Trough}_t}{\text{Peak}_t} \right)$$

### Why It Matters

Drawdowns measure the **pain** of investing, not just the statistical risk. A fund might have great average returns and a good Sharpe ratio, but if it lost 50% at some point, many investors would have panicked and sold at the bottom.

### Limitations

- Backward-looking: the worst drawdown in the past may not represent future risk.
- Sample-dependent: longer track records tend to show larger drawdowns.

> **Key Concept:** Maximum drawdown captures investor pain in a way that volatility cannot. A 50% drawdown requires a 100% gain just to break even.

> **CFA Exam Tip:** Maximum drawdown is a risk metric, not a return metric. It is particularly important for evaluating hedge funds and absolute return strategies.### Why Drawdown Matters

Standard deviation treats all volatility equally. But investors experience risk as *losses from a previous high*. A fund that drops 40% from its peak needs to gain 67% just to break even — that's the brutal math of drawdowns.

| Drawdown | Gain needed to recover |
|:--------:|:---------------------:|
| 10% | 11% |
| 20% | 25% |
| 30% | 43% |
| 50% | 100% |
| 75% | 300% |

> **Key Concept:** Drawdowns are asymmetric — the deeper the hole, the harder it is to climb out. This is why many risk-conscious investors focus on maximum drawdown as their primary risk metric, especially for absolute-return strategies.

Let's compute drawdowns for each fund and visualise the underwater chart:


In [ ]:
def max_drawdown(returns):
    """Compute maximum drawdown and its start/end indices."""
    wealth = np.cumprod(1 + returns)
    peak = np.maximum.accumulate(wealth)
    drawdown = (peak - wealth) / peak
    mdd = np.max(drawdown)
    end_idx = np.argmax(drawdown)
    start_idx = np.argmax(wealth[:end_idx + 1])
    return mdd, start_idx, end_idx, drawdown

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

for name, rets in funds.items():
    wealth = np.cumprod(1 + rets)
    mdd, s, e, dd = max_drawdown(rets)
    axes[0].plot(wealth, color=colors_f[name], linewidth=2, label=f"{name}")
    axes[1].fill_between(range(T), -dd * 100, 0, alpha=0.3, color=colors_f[name])
    axes[1].plot(-dd * 100, color=colors_f[name], linewidth=1, label=f"{name} MDD={mdd*100:.1f}%")

axes[0].set_ylabel('Wealth ($1 invested)')
axes[0].set_title('Cumulative Returns')
axes[0].legend()
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Month')
axes[1].set_title('Drawdown History')
axes[1].legend()
plt.tight_layout()
plt.show()

---
## 12. Value at Risk (VaR)

### What It Measures

The **maximum loss you would expect at a given confidence level** over a given time period. VaR at 95% answers: "I am 95% confident my loss will NOT exceed this amount."

### Three Estimation Methods

| Method | Approach | Pros | Cons |
|:--|:--|:--|:--|
| **Parametric** | Assume normal: $\text{VaR} = -(\mu + z_\alpha \sigma)$ | Fast, analytical | Assumes normality (fat tails!) |
| **Historical** | Take the empirical quantile | No distribution assumption | Needs lots of data |
| **Monte Carlo** | Simulate returns, take quantile | Flexible distributions | Computationally expensive |

### Worked Example (Parametric)

If monthly returns have $\mu = 0.8\%$ and $\sigma = 4\%$, the 95% VaR is:
$$\text{VaR}_{95\%} = -(0.8\% + (-1.645) \times 4\%) = -(0.8\% - 6.58\%) = 5.78\%$$

"I am 95% confident that this month's loss will not exceed 5.78%."

### The Famous Limitation of VaR

VaR tells you the threshold but says **NOTHING about how bad it gets beyond that threshold.** If VaR is 5%, your actual loss could be 5.1% or 50% -- VaR does not distinguish between these cases.

This is why VaR is sometimes called "a measure of how bad things can get -- except when they're really bad."

### Additional Limitations

1. **Not sub-additive:** The VaR of a combined portfolio can be LARGER than the sum of individual VaRs. This means VaR can penalize diversification -- an absurd property for a risk measure.
2. **Parametric VaR assumes normality:** Real returns have fat tails, so parametric VaR systematically underestimates extreme losses.

> **Key Concept:** VaR is a quantile-based risk measure. It tells you the loss at a specific confidence level but says nothing about losses beyond that point. Its main limitation is that it ignores the severity of tail losses.

> **CFA Exam Tip:** Know the three VaR methods (parametric, historical, Monte Carlo) and their tradeoffs. The key limitation to remember: VaR says nothing about HOW BAD losses can be beyond the threshold -- that is what CVaR/Expected Shortfall addresses.### Three Methods to Estimate VaR

| Method | Assumption | Pros | Cons |
|:-------|:-----------|:-----|:-----|
| **Parametric** | Returns are normal | Fast, analytical formula | Fails for fat-tailed distributions |
| **Historical** | Past returns represent future risk | No distribution assumption | Limited by sample size |
| **Monte Carlo** | Can use any model | Most flexible | Computationally expensive |

**Parametric VaR** (assuming normality):
$$\text{VaR}_{\alpha} = -(\mu + z_{\alpha} \cdot \sigma)$$

where $z_{\alpha}$ is the $\alpha$-quantile of the standard normal (e.g., $z_{0.05} = -1.645$).

**Historical VaR:** Simply sort all historical returns and pick the appropriate percentile.

> **Key Concept:** VaR answers "What is the MOST I could lose in 95% (or 99%) of months?" But it says NOTHING about how bad losses are in the remaining 5% (or 1%). You could lose 5% or 50% — VaR can't tell the difference. This is why we need CVaR/Expected Shortfall.

> **Common Mistake:** VaR is NOT the worst-case loss. It is a threshold. Actual losses can (and will) exceed VaR — the question is how often and by how much.

Let's compute VaR using all three methods:


In [ ]:
def var_parametric(returns, confidence=0.95):
    """Parametric VaR assuming normal distribution."""
    mu = np.mean(returns)
    sigma = np.std(returns)
    z = stats.norm.ppf(1 - confidence)
    return -(mu + z * sigma)

def var_historical(returns, confidence=0.95):
    """Historical VaR from empirical distribution."""
    return -np.percentile(returns, (1 - confidence) * 100)

def var_monte_carlo(returns, confidence=0.95, n_sims=100_000):
    """Monte Carlo VaR."""
    mu = np.mean(returns)
    sigma = np.std(returns)
    simulated = rng.normal(mu, sigma, n_sims)
    return -np.percentile(simulated, (1 - confidence) * 100)

confidence = 0.95
print(f"{'Fund':<10} {'Parametric':>12} {'Historical':>12} {'Monte Carlo':>12}")
print('-' * 48)
for name, rets in funds.items():
    vp = var_parametric(rets, confidence)
    vh = var_historical(rets, confidence)
    vm = var_monte_carlo(rets, confidence)
    print(f"{name:<10} {vp*100:>11.3f}% {vh*100:>11.3f}% {vm*100:>11.3f}%")

---
## 13. Conditional VaR / Expected Shortfall (CVaR)

### What It Measures

The **average loss when VaR is breached.** While VaR tells you the threshold, CVaR tells you what happens in the tail -- the expected loss on those really bad days.

### The Formula

$$\text{CVaR}_\alpha = E[\text{Loss} \mid \text{Loss} > \text{VaR}_\alpha] = -E[R \mid R < -\text{VaR}_\alpha]$$

### Why CVaR Is Better Than VaR

1. **CVaR is sub-additive:** The CVaR of a combined portfolio is always <= the sum of individual CVaRs. Diversification is never penalized.
2. **CVaR captures tail severity:** A fund with VaR of 5% and CVaR of 6% has a thin tail. A fund with VaR of 5% and CVaR of 15% has catastrophic tail risk -- even though their VaR is identical!
3. **CVaR is a coherent risk measure** (satisfies mathematical properties that a sensible risk measure should have). VaR is not coherent.

### Relationship: CVaR >= VaR Always

CVaR is always larger than VaR because it is the average of losses beyond the VaR threshold. If CVaR is much larger than VaR, the tail is fat and dangerous.

> **Key Concept:** CVaR (Expected Shortfall) answers: "When things go bad (beyond VaR), HOW bad do they get on average?" It is a more conservative and mathematically superior risk measure than VaR.

> **CFA Exam Tip:** CVaR is always >= VaR. It addresses VaR's main limitation by measuring tail severity. Regulators (Basel III) increasingly require CVaR in addition to VaR.### Why CVaR Is Superior to VaR

| Property | VaR | CVaR |
|:---------|:---:|:----:|
| Tells you the threshold? | Yes | Yes |
| Tells you expected loss beyond threshold? | No | Yes |
| Is a coherent risk measure? | No | Yes |
| Satisfies subadditivity? | No | Yes |

**Subadditivity** means: $\text{Risk}(A + B) \leq \text{Risk}(A) + \text{Risk}(B)$. Diversification should never *increase* measured risk. VaR can violate this; CVaR never does.

$$\text{CVaR}_{\alpha} = E[\text{Loss} \;|\; \text{Loss} > \text{VaR}_{\alpha}]$$

> **Key Concept:** CVaR (also called Expected Shortfall) answers: "When things go badly (beyond VaR), *how bad on average?*" This is exactly what risk managers and regulators care about — tail risk.

> **CFA Exam Tip:** Basel III banking regulations are shifting from VaR to Expected Shortfall for market risk capital requirements, precisely because ES is a coherent risk measure.

Let's compute CVaR and visualise the tail:


In [ ]:
def cvar_historical(returns, confidence=0.95):
    """Historical Conditional VaR (Expected Shortfall)."""
    cutoff = np.percentile(returns, (1 - confidence) * 100)
    tail = returns[returns <= cutoff]
    if len(tail) == 0:
        return -cutoff
    return -np.mean(tail)

def cvar_parametric(returns, confidence=0.95):
    """Parametric CVaR under normality."""
    mu = np.mean(returns)
    sigma = np.std(returns)
    z = stats.norm.ppf(1 - confidence)
    return -(mu - sigma * stats.norm.pdf(z) / (1 - confidence))

print(f"{'Fund':<10} {'VaR 95%':>10} {'CVaR 95%':>10}")
print('-' * 32)
for name, rets in funds.items():
    v = var_historical(rets, 0.95)
    cv = cvar_historical(rets, 0.95)
    print(f"{name:<10} {v*100:>9.3f}% {cv*100:>9.3f}%")

In [ ]:
# VaR/CVaR visualisation for Fund A
rets = fund_a
var_val = var_historical(rets, 0.95)
cvar_val = cvar_historical(rets, 0.95)

fig, ax = plt.subplots()
ax.hist(rets * 100, bins=25, density=True, color=PRIMARY, edgecolor='white', alpha=0.7)
ax.axvline(-var_val * 100, color=SECONDARY, linewidth=2, label=f'VaR 95% = {var_val*100:.2f}%')
ax.axvline(-cvar_val * 100, color=ACCENT, linewidth=2, linestyle='--', label=f'CVaR 95% = {cvar_val*100:.2f}%')
ax.fill_betweenx([0, ax.get_ylim()[1] * 0.8], -15, -var_val * 100, alpha=0.15, color=SECONDARY)
ax.set_xlabel('Monthly Return (%)')
ax.set_ylabel('Density')
ax.set_title('Fund A Return Distribution with VaR & CVaR')
ax.legend()
plt.tight_layout()
plt.show()

### Interpreting the VaR/CVaR Plot

- The **VaR line** (solid) marks the 5th percentile threshold. 95% of months have returns to the right of this line.
- The **CVaR line** (dashed) marks the average of the worst 5% of returns -- it is always further left (worse) than VaR.
- The **shaded area** represents the tail: the region where VaR is breached.
- The gap between VaR and CVaR tells you how dangerous the tail is. A large gap means occasional catastrophic losses.The key insight from this visualisation: the **red shaded area** (losses beyond VaR) is where the real danger lies. VaR tells you where this area starts; CVaR tells you the average depth of this red zone.

> **Key Concept:** During the 2008 financial crisis, many banks had VaR models suggesting daily losses would rarely exceed $50M. When Lehman Brothers collapsed, some experienced losses of $500M+ in a single day — 10x their VaR. This is why regulators now prefer Expected Shortfall.


---
## 14. Comparison Dashboard

Let's bring all metrics together to compare our three funds side by side. This is how practitioners evaluate funds in the real world -- never with just one metric, but with a complete profile.This is where the power of multiple measures becomes clear. No single metric tells the full story. Let's build a comprehensive dashboard:


In [ ]:
# Comprehensive comparison
metrics = {}
for name, rets in funds.items():
    alpha, beta = jensens_alpha(rets, benchmark, rf_monthly)
    mdd_val = max_drawdown(rets)[0]
    metrics[name] = {
        'Ann. Return': annualise_return(geometric_mean(rets)) * 100,
        'Ann. Vol': annualise_vol(np.std(rets)) * 100,
        'Sharpe': sharpe_ratio(rets, rf_monthly) * np.sqrt(12),
        'Sortino': sortino_ratio(rets, rf_monthly) * np.sqrt(12),
        'Treynor': treynor_ratio(rets, benchmark, rf_monthly)[0] * 12 * 100,
        'Alpha (ann)': alpha * 12 * 100,
        'Beta': beta,
        'Info Ratio': information_ratio(rets, benchmark) * np.sqrt(12),
        'Max DD': mdd_val * 100,
        'VaR 95%': var_historical(rets) * 100,
        'CVaR 95%': cvar_historical(rets) * 100,
    }

# Print table
metric_names = list(list(metrics.values())[0].keys())
header = f"{'Metric':<16}" + ''.join(f'{name:>12}' for name in funds.keys())
print(header)
print('=' * len(header))
for m in metric_names:
    vals = [metrics[name][m] for name in funds.keys()]
    if m == 'Beta':
        row = f"{m:<16}" + ''.join(f'{v:>12.3f}' for v in vals)
    elif m in ['Ann. Return', 'Ann. Vol', 'Max DD', 'VaR 95%', 'CVaR 95%', 'Treynor', 'Alpha (ann)']:
        row = f"{m:<16}" + ''.join(f'{v:>11.2f}%' for v in vals)
    else:
        row = f"{m:<16}" + ''.join(f'{v:>12.3f}' for v in vals)
    print(row)

In [ ]:
# Radar chart comparison
from matplotlib.patches import FancyBboxPatch

# Normalise selected metrics to [0, 1] for visual comparison
radar_metrics = ['Sharpe', 'Sortino', 'Info Ratio', 'Alpha (ann)', 'Max DD']
n_metrics = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for name in funds.keys():
    values = [metrics[name][m] for m in radar_metrics]
    # Invert Max DD (lower is better)
    values[-1] = -values[-1]
    # Normalise
    values_norm = [(v - min(metrics[n][m] if m != 'Max DD' else -metrics[n]['Max DD'] 
                   for n in funds)) / 
                   (max(metrics[n][m] if m != 'Max DD' else -metrics[n]['Max DD'] 
                   for n in funds) - 
                    min(metrics[n][m] if m != 'Max DD' else -metrics[n]['Max DD'] 
                   for n in funds) + 1e-10)
                   for v, m in zip(values, radar_metrics)]
    values_norm += values_norm[:1]
    ax.plot(angles, values_norm, 'o-', linewidth=2, color=colors_f[name], label=name)
    ax.fill(angles, values_norm, alpha=0.1, color=colors_f[name])

ax.set_xticks(angles[:-1])
labels = radar_metrics.copy()
labels[-1] = 'Low Drawdown'
ax.set_xticklabels(labels)
ax.set_title('Risk-Return Profile Comparison', pad=20)
ax.legend(loc='lower right', bbox_to_anchor=(1.3, 0))
plt.tight_layout()
plt.show()

### How to Read the Dashboard

No single fund "wins" on every dimension. This is typical in practice:

- **High-return funds** (Fund A) tend to have high volatility and large drawdowns.
- **Low-volatility funds** (Fund B) have lower tail risk but may not generate as much alpha.
- **Skilled managers** (Fund C) may have moderate returns but the best alpha and information ratio.

The right choice depends on your investment objectives, risk tolerance, and how the fund fits into your overall portfolio.

> **CFA Exam Tip:** In practice (and on the exam), fund evaluation requires looking at multiple metrics simultaneously. Never evaluate a fund on one number alone.### The Trade-offs in Practice

| If you care most about... | Look at... |
|:--------------------------|:-----------|
| Overall risk-adjusted performance | Sharpe ratio |
| Downside protection | Sortino ratio, Max drawdown |
| Market-relative performance | Alpha, Information ratio |
| Systematic risk efficiency | Treynor ratio |
| Tail risk | CVaR / Expected Shortfall |

> **Key Concept:** The "best" fund depends on what the investor cares about. A pension fund might prioritise low drawdowns. A hedge fund investor might focus on alpha. A diversified investor adding a satellite position should look at Treynor ratio. There is no single "best" metric.


---
## 15. References

1. Sharpe, W. F. "Mutual Fund Performance," *Journal of Business*, 1966.
2. Sortino, F. A. & Price, L. N. "Performance Measurement in a Downside Risk Framework," *Journal of Investing*, 1994.
3. Jensen, M. C. "The Performance of Mutual Funds in the Period 1945-1964," *Journal of Finance*, 1968.
4. Jorion, P. *Value at Risk*, 3rd ed., McGraw-Hill, 2007.
5. CFA Institute, *CFA Program Curriculum Level I -- Portfolio Management*.### CFA Level 1 Curriculum Alignment

The topics in this notebook map to the CFA Level 1 reading on "Risk and Return":
- LOS: Calculate and interpret major return measures (HPR, TWR, MWR, arithmetic, geometric)
- LOS: Describe the components of required return (risk-free rate + risk premiums)
- LOS: Explain risk aversion and its implications for portfolio selection (utility function)
- LOS: Calculate and interpret the Sharpe ratio, Treynor ratio, and Jensen's alpha
- LOS: Explain VaR and its limitations
